# Notebook 1: Data Scraping & Feature Engineering
## March Madness 2026 Projection System
---

**Data sources:**
1. **BartTorvik** — Team-level efficiency ratings (AdjOE, AdjDE, Barthag, tempo, four factors)
2. **Sports Reference** — Basic season stats (W-L, SRS, SOS, shooting, rebounds, turnovers)
3. **Sports Reference Advanced** — Pace, ORtg, eFG%, TOV%, ORB%, FT/FGA
4. **Sports Reference Historical Tournaments** — Game-level results 2002–2025 for model training
5. **2026 Bracket** — Hard-coded from official NCAA bracket

**Outputs → `data/` directory:**
- `barttorvik_teams_2026.csv` — BartTorvik team ratings
- `barttorvik_teams_{year}.csv` — BartTorvik historical team ratings (2008–2025)
- `sref_basic_2026.csv` / `sref_adv_2026.csv` — Sports Reference current season
- `sref_basic_{year}.csv` / `sref_adv_{year}.csv` — Sports Reference historical
- `historical_tourney.csv` — All tournament games 2002–2025 with seeds, scores, winners
- `historical_features.csv` — Pairwise matchup features for model training (historical)
- `bracket_2026.json` — Full 68-team bracket with regions, seeds, records
- `features_2026.csv` — Merged feature matrix for all 68 tournament teams

In [1]:
# ============================================================
# 1.0 CONFIG & IMPORTS
# ============================================================
import os, sys, time, json, warnings, logging, re
import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from pathlib import Path
from io import StringIO
from datetime import datetime

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("scraper")

# --- Paths ---
DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)

# --- Scraping config ---
SEASON = 2026
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )
}
RATE_LIMIT_SEC = 3.5   # polite delay between requests
CACHE_RAW = True       # save raw HTML for debugging
CACHE_MAX_AGE_HRS = 12 # re-scrape if cache older than this

# Historical range for training data
HIST_START = 2002  # was 2008, SportsRef data goes back
HIST_END = 2025

# --- Reproducibility ---
RANDOM_SEED = 51
np.random.seed(RANDOM_SEED)

log.info(f"Config: season={SEASON}, hist={HIST_START}-{HIST_END}, data_dir={DATA_DIR}")

2026-03-18 21:46:32,703 [INFO] Config: season=2026, hist=2002-2025, data_dir=data


In [2]:
# ============================================================
# 1.1 SCRAPING HELPERS
# ============================================================
def fetch_page(url, label="page", params=None):
    """Fetch URL with rate-limiting, retries, caching."""
    cache_path = DATA_DIR / f"raw_{label}.html"

    if CACHE_RAW and cache_path.exists():
        age_hrs = (time.time() - cache_path.stat().st_mtime) / 3600
        if age_hrs < CACHE_MAX_AGE_HRS:
            log.info(f"  Cache hit: {label} ({age_hrs:.1f}h old)")
            return cache_path.read_text(encoding="utf-8", errors="replace")

    for attempt in range(3):
        try:
            time.sleep(RATE_LIMIT_SEC)
            resp = requests.get(url, headers=HEADERS, params=params, timeout=20)
            resp.raise_for_status()
            html = resp.text
            if CACHE_RAW:
                cache_path.write_text(html, encoding="utf-8")
            log.info(f"  Fetched {label} ({len(html):,} chars)")
            return html
        except requests.RequestException as e:
            wait = 5 * (attempt + 1)
            log.warning(f"  Attempt {attempt+1}/3 failed for {label}: {e}. Retrying in {wait}s...")
            time.sleep(wait)

    raise RuntimeError(f"Failed to fetch {label} after 3 attempts: {url}")


def fetch_csv(url, label="csv", params=None):
    """Fetch a CSV endpoint directly. Returns raw text."""
    cache_path = DATA_DIR / f"raw_{label}.csv"

    if cache_path.exists():
        age_hrs = (time.time() - cache_path.stat().st_mtime) / 3600
        if age_hrs < CACHE_MAX_AGE_HRS:
            log.info(f"  Cache hit: {label} ({age_hrs:.1f}h old)")
            return cache_path.read_text(encoding="utf-8", errors="replace")

    for attempt in range(3):
        try:
            time.sleep(RATE_LIMIT_SEC)
            resp = requests.get(url, headers=HEADERS, params=params, timeout=20)
            resp.raise_for_status()
            text = resp.text
            cache_path.write_text(text, encoding="utf-8")
            log.info(f"  Fetched CSV {label} ({len(text):,} chars)")
            return text
        except requests.RequestException as e:
            wait = 5 * (attempt + 1)
            log.warning(f"  Attempt {attempt+1}/3 for {label}: {e}. Retrying...")
            time.sleep(wait)

    raise RuntimeError(f"Failed to fetch CSV {label}: {url}")


def safe_float(val, default=np.nan):
    """Convert scraped value to float safely."""
    if val is None:
        return default
    try:
        cleaned = str(val).strip().replace("+", "").replace("%", "")
        if cleaned in ("", "-", "—"):
            return default
        return float(cleaned)
    except (ValueError, TypeError):
        return default


def clean_team_name(name):
    """Normalize team name for cross-source joining."""
    if not isinstance(name, str):
        return ""
    name = name.strip()
    # Remove NCAA tournament marker from Sports Reference
    name = name.replace("NCAA", "").strip()
    # Lowercase for matching
    return name.lower().strip()


# Canonical name mapping: bracket name → Sports Reference name
# This resolves abbreviations, alternate names, and formatting differences.
# Add entries here if new mismatches appear.
TEAM_NAME_MAP = {
    # Abbreviations SRef uses full names for
    "tcu": "texas christian",
    "uconn": "connecticut",
    "byu": "brigham young",
    "vcu": "virginia commonwealth",
    "smu": "southern methodist",
    "ucf": "ucf",
    "umbc": "maryland-baltimore county",
    "lsu": "louisiana state",
    "unlv": "nevada-las vegas",
    "utep": "texas-el paso",
    "uab": "alabama-birmingham",
    # Naming differences
    "cal baptist": "california baptist",
    "long island": "long island university",
    "penn": "pennsylvania",
    "queens (n.c.)": "queens (nc)",
    "miami (fl)": "miami (fl)",
    "miami (ohio)": "miami (oh)",
    "st. john\'s": "st. john\'s (ny)",
    "saint mary\'s": "saint mary\'s",
    "ohio st.": "ohio state",
    "michigan st.": "michigan state",
    "iowa st.": "iowa state",
    "utah st.": "utah state",
    "north dakota st.": "north dakota state",
    "kennesaw st.": "kennesaw state",
    "wright st.": "wright state",
    "tennessee st.": "tennessee state",
    "south florida": "south florida",
    "north carolina": "north carolina",
    "texas a&m": "texas a&m",
    "texas tech": "texas tech",
    "nc state": "nc state",
    "prairie view a&m": "prairie view a&m",
    # Play-in composite names (map to first team; update after First Four)
    "nc state/texas": "nc state",
    "lehigh/pvamu": "lehigh",
    "howard/umbc": "howard",
    "smu/miami (oh)": "southern methodist",
}


def match_team(bracket_name, sref_names_lower):
    """
    Match a bracket team name to a Sports Reference school name.
    Returns the original-case SRef name, or None if no match.
    """
    bn = clean_team_name(bracket_name)

    # Direct match
    if bn in sref_names_lower:
        return sref_names_lower[bn]

    # Try canonical map
    mapped = TEAM_NAME_MAP.get(bn)
    if mapped and mapped in sref_names_lower:
        return sref_names_lower[mapped]

    # Fallback: substring match on first significant word
    words = bn.replace(".", "").split()
    if words:
        for sref_low, sref_orig in sref_names_lower.items():
            if words[0] in sref_low and len(words[0]) >= 4:
                return sref_orig

    return None

log.info("Helpers loaded")

2026-03-18 21:46:32,709 [INFO] Helpers loaded


In [3]:
# ============================================================
# 1.2 BARTTORVIK SCRAPER (TEAM-LEVEL)
# ============================================================
# BartTorvik data endpoints (per Bart's blog):
#   1. barttorvik.com/{YYYY}_team_results.csv — full team stats (has header row)
#   2. teamslicejson.php?year=YYYY&csv=1 — team slice data
#   3. getadvstats.php?year=YYYY&csv=1 — player-level fallback
#
# NOTE: trank.php is behind Cloudflare. The API endpoints above are not.

def scrape_barttorvik_teams(season):
    """Scrape BartTorvik TEAM-level efficiency ratings."""
    label = f"bt_teams_{season}"
    out_path = DATA_DIR / f"barttorvik_teams_{season}.csv"

    if out_path.exists():
        age_hrs = (time.time() - out_path.stat().st_mtime) / 3600
        if age_hrs < CACHE_MAX_AGE_HRS:
            log.info(f"  Cache hit: {label}")
            return pd.read_csv(out_path)

    log.info(f"Scraping BartTorvik teams for {season}...")

    # --- Attempt 1: {YYYY}_team_results.csv ---
    try:
        url1 = f"https://barttorvik.com/{season}_team_results.csv"
        text = fetch_csv(url1, label=f"{label}_results")

        if text.strip().startswith("<!") or text.strip().startswith("<html"):
            raise ValueError("Got HTML instead of CSV")

        # File has its own header row. Try comma first, then tab.
        df = None
        for sep in [",", "\t", ";"]:
            try:
                candidate = pd.read_csv(StringIO(text), sep=sep, header=0, on_bad_lines="skip")
                if len(candidate) > 200 and candidate.shape[1] >= 10:
                    df = candidate
                    break
            except Exception:
                continue

        if df is not None and len(df) > 200:
            # Clean column names (lowercase from file, normalize)
            df.columns = [str(c).strip() for c in df.columns]
            # Standardize common column names
            col_map = {
                "team": "Team", "conf": "Conf",
                "adjoe": "AdjOE", "adjde": "AdjDE",
                "barthag": "Barthag", "adj t.": "AdjTempo",
                "adjt": "AdjTempo", "adj_t": "AdjTempo",
                "wab": "WAB", "seed": "Seed",
                "efg%": "EFG_O", "efg% d": "EFG_D",
                "ft rate": "FT_Rate", "ft rate d": "FT_Rate_D",
                "to%": "TO_Rate", "to% d": "TO_Rate_D",
                "oreb%": "OReb_Rate", "dreb%": "DReb_Rate",
                "2pt%": "TwoP_O", "2pt% d": "TwoP_D",
                "3pt%": "ThreeP_O", "3pt% d": "ThreeP_D",
                "3pt rate": "ThreePRate", "3pt rate d": "ThreePRate_D",
            }
            df.columns = [col_map.get(c.lower(), c) for c in df.columns]

            # Detect column misalignment: BT team_results.csv for 2008-2022 has 44 cols
            # with NO rank header, so pandas shifts all headers left by 1.
            # Result: 'rank' col has team names, 'Team' col has conferences, etc.
            # Fix: if first value of 'rank' column is a string (team name), reassign headers.
            first_val = str(df.iloc[0][df.columns[0]])
            is_shifted = (first_val and not first_val.replace('.','').replace('-','').isdigit() 
                         and len(first_val) > 2)
            
            if is_shifted:
                # Headers are shifted left by 1. The correct order (from 2023+ files) is:
                # Team, Conf, record, AdjOE, oe Rank, AdjDE, de Rank, Barthag, ...
                correct_headers = [
                    "Team", "Conf", "record", "AdjOE", "oe Rank", "AdjDE", "de Rank",
                    "Barthag", "rank.1", "proj. W", "Proj. L", "Pro Con W", "Pro Con L",
                    "Con Rec.", "sos", "ncsos", "consos", "Proj. SOS", "Proj. Noncon SOS",
                    "Proj. Con SOS", "elite SOS", "elite noncon SOS", "Opp OE", "Opp DE",
                    "Opp Proj. OE", "Opp Proj DE", "Con Adj OE", "Con Adj DE",
                    "Qual O", "Qual D", "Qual Barthag", "Qual Games", "FUN",
                    "ConPF", "ConPA", "ConPoss", "ConOE", "ConDE", "ConSOSRemain",
                    "Conf Win%", "WAB", "WAB Rk", "Fun Rk", "AdjTempo",
                ]
                if len(correct_headers) >= len(df.columns):
                    df.columns = correct_headers[:len(df.columns)]
                else:
                    df.columns = correct_headers + [f"Extra_{i}" for i in range(len(df.columns) - len(correct_headers))]
                log.info(f"  Fixed column shift: Team='{df['Team'].iloc[0]}', AdjOE={df['AdjOE'].iloc[0]:.1f}")

            log.info(f"  ✅ team_results.csv: {len(df)} teams, {df.shape[1]} cols")
            log.info(f"     Columns: {list(df.columns[:15])}...")
            df.to_csv(out_path, index=False)
            return df
        else:
            log.warning(f"  team_results.csv parsed but too small: {df.shape if df is not None else 'None'}")
    except Exception as e:
        log.warning(f"  team_results.csv failed: {e}")

    # --- Attempt 2: teamslicejson.php?csv=1 ---
    try:
        url2 = "https://barttorvik.com/teamslicejson.php"
        params2 = {"year": season, "csv": 1}
        text2 = fetch_csv(url2, label=f"{label}_slice", params=params2)

        if text2.strip().startswith("<!") or text2.strip().startswith("<html"):
            raise ValueError("Got HTML instead of CSV")

        # teamslicejson may or may not have headers — try both
        for header_mode in [0, None]:
            try:
                for sep in [",", "\t"]:
                    candidate = pd.read_csv(StringIO(text2), sep=sep, header=header_mode, on_bad_lines="skip")
                    if len(candidate) > 200 and candidate.shape[1] >= 5:
                        df2 = candidate
                        if header_mode is None:
                            # No header — assign generic names
                            df2.columns = [f"col_{i}" for i in range(df2.shape[1])]
                            # First col is typically team name
                            df2.rename(columns={"col_0": "Team", "col_1": "Conf"}, inplace=True)
                        else:
                            df2.columns = [str(c).strip() for c in df2.columns]
                        log.info(f"  ✅ teamslicejson: {len(df2)} teams, {df2.shape[1]} cols")
                        df2.to_csv(out_path, index=False)
                        return df2
            except Exception:
                continue

        log.warning(f"  teamslicejson parsed but wrong shape")
    except Exception as e:
        log.warning(f"  teamslicejson failed: {e}")

    # --- Attempt 3: player-level → aggregate ---
    try:
        url3 = "https://barttorvik.com/getadvstats.php"
        params3 = {"year": season, "csv": 1}
        text3 = fetch_csv(url3, label=f"{label}_players", params=params3)

        if text3.strip().startswith("<!") or text3.strip().startswith("<html"):
            raise ValueError("Got HTML instead of CSV")

        df3 = pd.read_csv(StringIO(text3), header=None, on_bad_lines="skip")
        if len(df3) > 500:
            log.info(f"  Got player data ({len(df3)} rows). Aggregating...")
            df3 = _aggregate_player_to_team(df3)
            if len(df3) > 50:
                log.info(f"  ✅ Player→team: {len(df3)} teams")
                df3.to_csv(out_path, index=False)
                return df3
    except Exception as e:
        log.warning(f"  getadvstats failed: {e}")

    log.error(f"  ❌ No BartTorvik data for {season}")
    return None


def _aggregate_player_to_team(player_df):
    """Aggregate BartTorvik player CSV to team-level (minutes-weighted)."""
    df = player_df.copy()
    df.columns = [f"col_{i}" for i in range(df.shape[1])]
    team_col, conf_col = "col_1", "col_2"
    gp_col, minpct_col = "col_3", "col_4"
    ortg_col, usage_col, efg_col, ts_col = "col_5", "col_6", "col_7", "col_8"

    for c in [gp_col, minpct_col, ortg_col, usage_col, efg_col, ts_col]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["_weight"] = df[minpct_col].fillna(0)

    def wmean(x, w):
        mask = x.notna() & w.notna() & (w > 0)
        if mask.sum() == 0: return np.nan
        return np.average(x[mask], weights=w[mask])

    teams = df.groupby(team_col).apply(
        lambda g: pd.Series({
            "Conf": g[conf_col].iloc[0],
            "N_Players": len(g),
            "AvgORtg": wmean(g[ortg_col], g["_weight"]),
            "AvgUsage": wmean(g[usage_col], g["_weight"]),
            "AvgEFG": wmean(g[efg_col], g["_weight"]),
        })
    ).reset_index()
    teams.rename(columns={team_col: "Team"}, inplace=True)
    return teams


# --- Scrape current season ---
bart_2026 = scrape_barttorvik_teams(SEASON)
if bart_2026 is not None:
    print(f"\nBartTorvik {SEASON}: {bart_2026.shape}")
    key_cols = [c for c in ["Team", "Conf", "AdjOE", "AdjDE", "Barthag", "AdjTempo", "WAB"] if c in bart_2026.columns]
    if key_cols:
        # Skip the embedded header row if present
        display_df = bart_2026[bart_2026["Team"] != "team"] if "Team" in bart_2026.columns else bart_2026
        display(display_df[key_cols].head(10))
    else:
        display(bart_2026.head())
    print(f"\nAll columns: {list(bart_2026.columns)}")
else:
    print("⚠️ BartTorvik not available. Sports Reference will be primary source.")

2026-03-18 21:46:32,718 [INFO] Scraping BartTorvik teams for 2026...
2026-03-18 21:46:37,204 [INFO]   Fetched CSV bt_teams_2026_results (216,753 chars)
2026-03-18 21:46:37,221 [INFO]   ✅ team_results.csv: 365 teams, 45 cols
2026-03-18 21:46:37,221 [INFO]      Columns: ['rank', 'Team', 'Conf', 'record', 'AdjOE', 'oe Rank', 'AdjDE', 'de Rank', 'Barthag', 'rank.1', 'proj. W', 'Proj. L', 'Pro Con W', 'Pro Con L', 'Con Rec.']...



BartTorvik 2026: (365, 45)


,Team,Conf,AdjOE,AdjDE,Barthag,AdjTempo,WAB
0,Duke,ACC,128.098849,90.707837,0.981464,65.790933,13.612639
1,Michigan,B10,127.581651,90.918455,0.980084,71.205027,13.953091
2,Arizona,B12,126.839872,91.287482,0.977741,70.013361,13.606138
3,Florida,SEC,126.031142,92.193451,0.973283,70.385431,8.480105
4,Houston,B12,125.244173,92.298836,0.970976,63.315636,8.931423
5,Illinois,B10,131.891944,98.106328,0.967805,65.702815,6.535892
6,Iowa St.,B12,123.672556,92.521946,0.965683,66.940510,7.338254
7,Purdue,B10,133.265744,100.145386,0.963939,64.424606,9.143874
8,Connecticut,BE,123.039239,94.913577,0.951877,64.571066,8.795612
9,Vanderbilt,SEC,127.405629,99.269715,0.946326,68.971046,7.250852



All columns: ['rank', 'Team', 'Conf', 'record', 'AdjOE', 'oe Rank', 'AdjDE', 'de Rank', 'Barthag', 'rank.1', 'proj. W', 'Proj. L', 'Pro Con W', 'Pro Con L', 'Con Rec.', 'sos', 'ncsos', 'consos', 'Proj. SOS', 'Proj. Noncon SOS', 'Proj. Con SOS', 'elite SOS', 'elite noncon SOS', 'Opp OE', 'Opp DE', 'Opp Proj. OE', 'Opp Proj DE', 'Con Adj OE', 'Con Adj DE', 'Qual O', 'Qual D', 'Qual Barthag', 'Qual Games', 'FUN', 'ConPF', 'ConPA', 'ConPoss', 'ConOE', 'ConDE', 'ConSOSRemain', 'Conf Win%', 'WAB', 'WAB Rk', 'Fun Rk', 'AdjTempo']


In [4]:
# ============================================================
# 1.3 BARTTORVIK HISTORICAL (for training models)
# ============================================================
log.info("Scraping BartTorvik historical team data...")

bart_historical = {}
for year in range(HIST_START, HIST_END + 1):
    if year == 2020:
        continue
    try:
        df = scrape_barttorvik_teams(year)
        if df is not None and len(df) > 50:
            bart_historical[year] = df
            log.info(f"  {year}: {len(df)} teams ✅")
        else:
            log.warning(f"  {year}: no data ❌")
    except Exception as e:
        log.warning(f"  {year}: failed - {e}")

log.info(f"BartTorvik historical: {len(bart_historical)} seasons loaded")
print(f"\nSeasons with data: {sorted(bart_historical.keys())}")

# Show what columns we got (may differ by endpoint)
if bart_historical:
    sample_year = max(bart_historical.keys())
    sample_df = bart_historical[sample_year]
    print(f"\nSample columns ({sample_year}): {list(sample_df.columns)}")

2026-03-18 21:46:37,242 [INFO] Scraping BartTorvik historical team data...
2026-03-18 21:46:37,243 [INFO] Scraping BartTorvik teams for 2002...
2026-03-18 21:46:41,353 [INFO]   Fetched CSV bt_teams_2002_results (430 chars)
2026-03-18 21:46:41,354 [WARNING]   team_results.csv failed: Got HTML instead of CSV
2026-03-18 21:46:45,349 [INFO]   Fetched CSV bt_teams_2002_slice (52 chars)
2026-03-18 21:46:45,356 [WARNING]   teamslicejson parsed but wrong shape
2026-03-18 21:46:55,915 [INFO]   Fetched CSV bt_teams_2002_players (1,952,113 chars)
2026-03-18 21:46:55,951 [INFO]   Got player data (4979 rows). Aggregating...
2026-03-18 21:46:56,126 [INFO]   ✅ Player→team: 365 teams
2026-03-18 21:46:56,128 [INFO]   2002: 365 teams ✅
2026-03-18 21:46:56,128 [INFO] Scraping BartTorvik teams for 2003...
2026-03-18 21:47:00,482 [INFO]   Fetched CSV bt_teams_2003_results (430 chars)
2026-03-18 21:47:00,483 [WARNING]   team_results.csv failed: Got HTML instead of CSV
2026-03-18 21:47:04,491 [INFO]   Fetche


Seasons with data: [2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2021, 2022, 2023, 2024, 2025]

Sample columns (2025): ['rank', 'Team', 'Conf', 'record', 'AdjOE', 'oe Rank', 'AdjDE', 'de Rank', 'Barthag', 'rank.1', 'proj. W', 'Proj. L', 'Pro Con W', 'Pro Con L', 'Con Rec.', 'sos', 'ncsos', 'consos', 'Proj. SOS', 'Proj. Noncon SOS', 'Proj. Con SOS', 'elite SOS', 'elite noncon SOS', 'Opp OE', 'Opp DE', 'Opp Proj. OE', 'Opp Proj DE', 'Con Adj OE', 'Con Adj DE', 'Qual O', 'Qual D', 'Qual Barthag', 'Qual Games', 'FUN', 'ConPF', 'ConPA', 'ConPoss', 'ConOE', 'ConDE', 'ConSOSRemain', 'Conf Win%', 'WAB', 'WAB Rk', 'Fun Rk', 'AdjTempo']


In [5]:
# ============================================================
# 1.4 SPORTS REFERENCE — BASIC STATS
# ============================================================
def scrape_sref_basic(season):
    """
    Scrape sports-reference.com basic school stats.
    Handles the multi-row header and 'NCAA' suffix on tourney teams.
    """
    label = f"sref_basic_{season}"
    out_path = DATA_DIR / f"sref_basic_{season}.csv"

    if out_path.exists():
        age_hrs = (time.time() - out_path.stat().st_mtime) / 3600
        if age_hrs < CACHE_MAX_AGE_HRS:
            df = pd.read_csv(out_path)
            if len(df) > 100:
                log.info(f"  Cache hit: {label} ({len(df)} teams)")
                return df

    log.info(f"  Scraping Sports Ref basic for {season}...")
    url = f"https://www.sports-reference.com/cbb/seasons/men/{season}-school-stats.html"

    try:
        html = fetch_page(url, label=label)
        soup = BeautifulSoup(html, "html.parser")

        table = soup.find("table", {"id": "basic_school_stats"})
        if table is None:
            log.warning(f"  No basic_school_stats table for {season}")
            return None

        # Parse the second header row (actual column names)
        thead = table.find("thead")
        header_rows = thead.find_all("tr")
        headers = [th.get_text(strip=True) for th in header_rows[-1].find_all("th")]

        # Parse body — skip sub-header rows
        tbody = table.find("tbody")
        rows = []
        for tr in tbody.find_all("tr"):
            if "thead" in (tr.get("class") or []):
                continue
            cells_data = [cell.get_text(strip=True) for cell in tr.find_all(["td", "th"])]
            if cells_data and len(cells_data) == len(headers):
                rows.append(cells_data)

        df = pd.DataFrame(rows, columns=headers)

        # Drop empty separator columns
        df = df.loc[:, df.columns != ""]
        # Deduplicate column names
        cols = []
        seen = {}
        for c in df.columns:
            if c in seen:
                seen[c] += 1
                cols.append(f"{c}_{seen[c]}")
            else:
                seen[c] = 0
                cols.append(c)
        df.columns = cols

        # Mark NCAA tournament teams and clean names
        df["in_tourney"] = df["School"].str.contains("NCAA", na=False)
        df["School"] = df["School"].str.replace("NCAA", "", regex=False).str.strip()

        # Convert numeric columns
        for col in df.columns:
            if col not in ("School", "Rk", "in_tourney"):
                df[col] = pd.to_numeric(df[col], errors="coerce")

        df.to_csv(out_path, index=False)
        log.info(f"  Sports Ref basic {season}: {len(df)} teams, {df.shape[1]} cols")
        return df

    except Exception as e:
        log.error(f"  Sports Ref basic {season} failed: {e}")
        if out_path.exists():
            return pd.read_csv(out_path)
        return None


# Scrape current season
sref_basic = scrape_sref_basic(SEASON)
if sref_basic is not None:
    print(f"\nSRef Basic 2026: {sref_basic.shape}")
    # Show tourney teams
    tourney = sref_basic[sref_basic["in_tourney"]]
    print(f"Tournament teams found: {len(tourney)}")
    display(sref_basic.head())

2026-03-18 21:49:33,107 [INFO]   Scraping Sports Ref basic for 2026...
2026-03-18 21:49:36,788 [INFO]   Fetched sref_basic_2026 (917,016 chars)
2026-03-18 21:49:37,017 [INFO]   Sports Ref basic 2026: 365 teams, 34 cols



SRef Basic 2026: (365, 34)
Tournament teams found: 68


,Rk,School,G,W,L,W-L%,SRS,SOS,W_1,L_1,...,FTA,FT%,ORB,TRB,AST,STL,BLK,TOV,PF,in_tourney
0,1,Abilene Christian,33,14,19,0.424,-7.18,-1.25,5,13,...,741,0.704,369,1047,440,321,91,472,691,False
1,2,Air Force,32,3,29,0.094,-13.66,4.68,0,20,...,574,0.641,225,931,389,180,76,472,570,False
2,3,Akron,34,29,5,0.853,9.04,-3.33,17,1,...,601,0.754,386,1294,627,256,105,372,536,True
3,4,Alabama,32,23,9,0.719,22.70,14.45,13,5,...,784,0.765,391,1300,516,206,162,313,593,True
4,5,Alabama A&M,33,18,15,0.545,-12.99,-10.49,10,8,...,808,0.738,311,1126,387,198,93,376,607,False


In [6]:
# ============================================================
# 1.5 SPORTS REFERENCE — ADVANCED STATS
# ============================================================
def scrape_sref_advanced(season):
    """
    Scrape sports-reference.com advanced school stats.
    Key stats: Pace, ORtg, eFG%, TOV%, ORB%, FT/FGA
    """
    label = f"sref_adv_{season}"
    out_path = DATA_DIR / f"sref_adv_{season}.csv"

    if out_path.exists():
        age_hrs = (time.time() - out_path.stat().st_mtime) / 3600
        if age_hrs < CACHE_MAX_AGE_HRS:
            df = pd.read_csv(out_path)
            if len(df) > 100:
                log.info(f"  Cache hit: {label} ({len(df)} teams)")
                return df

    log.info(f"  Scraping Sports Ref advanced for {season}...")
    url = f"https://www.sports-reference.com/cbb/seasons/men/{season}-advanced-school-stats.html"

    try:
        html = fetch_page(url, label=label)
        soup = BeautifulSoup(html, "html.parser")

        table = soup.find("table", {"id": "adv_school_stats"})
        if table is None:
            log.warning(f"  No adv_school_stats table for {season}")
            return None

        thead = table.find("thead")
        header_rows = thead.find_all("tr")
        headers = [th.get_text(strip=True) for th in header_rows[-1].find_all("th")]

        tbody = table.find("tbody")
        rows = []
        for tr in tbody.find_all("tr"):
            if "thead" in (tr.get("class") or []):
                continue
            cells_data = [cell.get_text(strip=True) for cell in tr.find_all(["td", "th"])]
            if cells_data and len(cells_data) == len(headers):
                rows.append(cells_data)

        df = pd.DataFrame(rows, columns=headers)
        df = df.loc[:, df.columns != ""]

        # Deduplicate columns (same W/L appear under Overall, Conf, Home, Away)
        cols = []
        seen = {}
        for c in df.columns:
            if c in seen:
                seen[c] += 1
                cols.append(f"{c}_{seen[c]}")
            else:
                seen[c] = 0
                cols.append(c)
        df.columns = cols

        df["in_tourney"] = df["School"].str.contains("NCAA", na=False)
        df["School"] = df["School"].str.replace("NCAA", "", regex=False).str.strip()

        for col in df.columns:
            if col not in ("School", "Rk", "in_tourney"):
                df[col] = pd.to_numeric(df[col], errors="coerce")

        df.to_csv(out_path, index=False)
        log.info(f"  Sports Ref adv {season}: {len(df)} teams, {df.shape[1]} cols")
        return df

    except Exception as e:
        log.error(f"  Sports Ref adv {season} failed: {e}")
        if out_path.exists():
            return pd.read_csv(out_path)
        return None


sref_adv = scrape_sref_advanced(SEASON)
if sref_adv is not None:
    print(f"\nSRef Advanced 2026: {sref_adv.shape}")
    display(sref_adv.head())

2026-03-18 21:49:37,027 [INFO]   Scraping Sports Ref advanced for 2026...
2026-03-18 21:49:40,662 [INFO]   Fetched sref_adv_2026 (901,745 chars)
2026-03-18 21:49:40,874 [INFO]   Sports Ref adv 2026: 365 teams, 30 cols



SRef Advanced 2026: (365, 30)


,Rk,School,G,W,L,W-L%,SRS,SOS,W_1,L_1,...,TS%,TRB%,AST%,STL%,BLK%,eFG%,TOV%,ORB%,FT/FGA,in_tourney
0,1,Abilene Christian,33,14,19,0.424,-7.18,-1.25,5,13,...,0.529,50.4,54.7,14.0,8.1,0.489,17.7,32.6,0.284,False
1,2,Air Force,32,3,29,0.094,-13.66,4.68,0,20,...,0.516,44.8,55.9,8.3,6.8,0.489,19.8,21.4,0.225,False
2,3,Akron,34,29,5,0.853,9.04,-3.33,17,1,...,0.612,53.5,57.5,10.5,9.1,0.588,13.2,34.0,0.209,True
3,4,Alabama,32,23,9,0.719,22.70,14.45,13,5,...,0.592,50.4,53.5,8.6,11.3,0.554,11.2,31.8,0.285,True
4,5,Alabama A&M,33,18,15,0.545,-12.99,-10.49,10,8,...,0.545,49.9,49.4,8.8,7.9,0.496,14.7,28.0,0.331,False


In [7]:
# ============================================================
# 1.6 SPORTS REFERENCE — HISTORICAL STATS (for training)
# ============================================================
log.info("Scraping Sports Reference historical data...")

sref_basic_hist = {}
sref_adv_hist = {}

for year in range(HIST_START, HIST_END + 1):
    if year == 2020:
        continue
    try:
        basic = scrape_sref_basic(year)
        if basic is not None and len(basic) > 100:
            sref_basic_hist[year] = basic

        adv = scrape_sref_advanced(year)
        if adv is not None and len(adv) > 100:
            sref_adv_hist[year] = adv

        log.info(f"  {year}: basic={len(basic) if basic is not None else 0}, "
                 f"adv={len(adv) if adv is not None else 0}")
    except Exception as e:
        log.warning(f"  {year} failed: {e}")

log.info(f"Historical SRef: {len(sref_basic_hist)} seasons basic, "
         f"{len(sref_adv_hist)} seasons advanced")

2026-03-18 21:49:40,881 [INFO] Scraping Sports Reference historical data...
2026-03-18 21:49:40,882 [INFO]   Scraping Sports Ref basic for 2002...
2026-03-18 21:49:44,518 [INFO]   Fetched sref_basic_2002 (826,856 chars)
2026-03-18 21:49:44,728 [INFO]   Sports Ref basic 2002: 323 teams, 34 cols
2026-03-18 21:49:44,729 [INFO]   Scraping Sports Ref advanced for 2002...
2026-03-18 21:49:48,392 [INFO]   Fetched sref_adv_2002 (810,941 chars)
2026-03-18 21:49:48,580 [INFO]   Sports Ref adv 2002: 323 teams, 30 cols
2026-03-18 21:49:48,580 [INFO]   2002: basic=323, adv=323
2026-03-18 21:49:48,581 [INFO]   Scraping Sports Ref basic for 2003...
2026-03-18 21:49:52,244 [INFO]   Fetched sref_basic_2003 (834,204 chars)
2026-03-18 21:49:52,449 [INFO]   Sports Ref basic 2003: 327 teams, 34 cols
2026-03-18 21:49:52,449 [INFO]   Scraping Sports Ref advanced for 2003...
2026-03-18 21:49:56,084 [INFO]   Fetched sref_adv_2003 (817,882 chars)
2026-03-18 21:49:56,277 [INFO]   Sports Ref adv 2003: 327 teams, 

In [8]:
# ============================================================
# 1.7 HISTORICAL TOURNAMENT RESULTS (2002–2025)
# ============================================================
def scrape_tournament_results(season):
    """
    Scrape a single season's tournament results from Sports Reference.
    
    SR structure (verified against actual HTML):
    - div#brackets contains 5 div#bracket elements:
      - Brackets 0-3: one per region (8+4+2+1=15 games each, 5 rounds)
      - Bracket 4: Final Four + Championship (3 games, 3 rounds)
    - Each bracket has div.round children
    - Each round has game divs containing two team divs
    - Team div: <span>seed</span> <a>team_name</a> <a>score</a>
    - Winner's div has class="winner"
    """
    label = f"tourney_{season}"
    url = f"https://www.sports-reference.com/cbb/postseason/men/{season}-ncaa.html"

    try:
        html = fetch_page(url, label=label)
        soup = BeautifulSoup(html, "html.parser")

        # Find the brackets container (holds all 5 bracket divs)
        brackets_container = soup.find("div", {"id": "brackets"})
        if brackets_container is None:
            log.warning(f"  No #brackets container for {season}")
            return []

        bracket_divs = brackets_container.find_all("div", {"id": "bracket"})
        if not bracket_divs:
            log.warning(f"  No bracket divs found for {season}")
            return []

        games = []
        round_names_regional = ["R64", "R32", "S16", "E8", "E8"]  # 5 rounds, last is regional final
        round_names_ff = ["F4", "Championship", "Championship"]

        for bi, brack in enumerate(bracket_divs):
            if bi < 4:
                round_map = round_names_regional
            else:
                round_map = round_names_ff

            rounds = brack.find_all("div", class_="round")

            for rnd_idx, rnd in enumerate(rounds):
                rnd_label = round_map[rnd_idx] if rnd_idx < len(round_map) else f"R{rnd_idx}"

                # Map to numeric round for feature engineering
                round_num_map = {"R64": 1, "R32": 2, "S16": 3, "E8": 4, "F4": 5, "Championship": 6}
                rnd_num = round_num_map.get(rnd_label, rnd_idx + 1)

                game_divs = rnd.find_all("div", recursive=False)

                for game_div in game_divs:
                    team_divs = game_div.find_all("div", recursive=False)
                    if len(team_divs) < 2:
                        continue

                    team_data = []
                    for tdiv in team_divs[:2]:
                        seed_span = tdiv.find("span", recursive=False)
                        seed = None
                        if seed_span:
                            seed = safe_float(seed_span.get_text(strip=True))

                        links = tdiv.find_all("a")
                        team_name = None
                        score = None
                        if len(links) >= 1:
                            team_name = links[0].get_text(strip=True)
                        if len(links) >= 2:
                            score = safe_float(links[1].get_text(strip=True))

                        is_winner = "winner" in (tdiv.get("class") or [])

                        team_data.append({
                            "name": team_name,
                            "seed": seed,
                            "score": score,
                            "is_winner": is_winner,
                        })

                    if len(team_data) == 2 and team_data[0]["name"] and team_data[1]["name"]:
                        games.append({
                            "season": season,
                            "round": rnd_label,
                            "round_num": rnd_num,
                            "team1": team_data[0]["name"],
                            "seed1": team_data[0]["seed"],
                            "score1": team_data[0]["score"],
                            "team2": team_data[1]["name"],
                            "seed2": team_data[1]["seed"],
                            "score2": team_data[1]["score"],
                            "winner": (team_data[0]["name"] if team_data[0]["is_winner"]
                                       else (team_data[1]["name"] if team_data[1]["is_winner"]
                                             else None)),
                        })

        return games

    except Exception as e:
        log.error(f"  Tournament {season} failed: {e}")
        return []


# Scrape all historical tournaments
log.info("Scraping historical tournament results 2002-2025...")
all_tourney_games = []

for year in range(2002, 2026):
    if year == 2020:  # cancelled
        continue
    games = scrape_tournament_results(year)
    all_tourney_games.extend(games)
    n = len(games)
    status = "✅" if n == 63 else ("⚠️" if n > 0 else "❌")
    log.info(f"  {year}: {n} games {status}")

hist_df = pd.DataFrame(all_tourney_games)

# Derive winner_idx
if len(hist_df) > 0:
    hist_df["winner_idx"] = np.where(
        hist_df["winner"] == hist_df["team1"], 1,
        np.where(hist_df["winner"] == hist_df["team2"], 2, np.nan)
    )
    hist_df["score_diff"] = hist_df["score1"] - hist_df["score2"]

    hist_df.to_csv(DATA_DIR / "historical_tourney.csv", index=False)
    log.info(f"\nHistorical data: {len(hist_df)} games, "
             f"{hist_df['season'].nunique()} seasons")
    
    print(f"\nGames per round:")
    print(hist_df["round"].value_counts().sort_index().to_string())
    print(f"\nExpected: 23 seasons × 63 games = {23*63} games")
    print(f"Actual: {len(hist_df)} games")
    print(f"\nSample (2025 Final Four + Championship):")
    display(hist_df[
        (hist_df["season"] == 2025) & 
        (hist_df["round"].isin(["F4", "Championship"]))
    ])
else:
    log.error("No historical tournament data scraped!")

2026-03-18 21:52:38,631 [INFO] Scraping historical tournament results 2002-2025...
2026-03-18 21:52:42,247 [INFO]   Fetched tourney_2002 (172,078 chars)
2026-03-18 21:52:42,280 [INFO]   2002: 63 games ✅
2026-03-18 21:52:45,931 [INFO]   Fetched tourney_2003 (171,193 chars)
2026-03-18 21:52:45,969 [INFO]   2003: 63 games ✅
2026-03-18 21:52:49,586 [INFO]   Fetched tourney_2004 (172,033 chars)
2026-03-18 21:52:49,654 [INFO]   2004: 63 games ✅
2026-03-18 21:52:53,275 [INFO]   Fetched tourney_2005 (171,999 chars)
2026-03-18 21:52:53,311 [INFO]   2005: 63 games ✅
2026-03-18 21:52:56,933 [INFO]   Fetched tourney_2006 (171,825 chars)
2026-03-18 21:52:56,970 [INFO]   2006: 63 games ✅
2026-03-18 21:53:00,578 [INFO]   Fetched tourney_2007 (171,877 chars)
2026-03-18 21:53:00,612 [INFO]   2007: 63 games ✅
2026-03-18 21:53:04,222 [INFO]   Fetched tourney_2008 (171,310 chars)
2026-03-18 21:53:04,256 [INFO]   2008: 63 games ✅
2026-03-18 21:53:07,866 [INFO]   Fetched tourney_2009 (171,823 chars)
2026-03


Games per round:
round
Championship     23
E8               92
F4               46
R32             368
R64             736
S16             184

Expected: 23 seasons × 63 games = 1449 games
Actual: 1449 games

Sample (2025 Final Four + Championship):


,season,round,round_num,team1,seed1,score1,team2,seed2,score2,winner,winner_idx,score_diff
1446,2025,F4,5,Duke,1.0,67.0,Houston,1.0,70.0,Houston,2.0,-3.0
1447,2025,F4,5,Auburn,1.0,73.0,Florida,1.0,79.0,Florida,2.0,-6.0
1448,2025,Championship,6,Houston,1.0,63.0,Florida,1.0,65.0,Florida,2.0,-2.0


In [9]:
# ============================================================
# 1.8 2026 BRACKET (HARD-CODED FROM NCAA.COM)
# ============================================================
# Source: Official 2026 NCAA bracket released 3/16/2026
# First Four games TBD (play 3/17-3/18 in Dayton)

bracket_2026 = [
    # ===== EAST REGION =====
    {"region": "East", "seed": 1,  "team": "Duke",             "record": "32-2"},
    {"region": "East", "seed": 16, "team": "Siena",            "record": "23-11"},
    {"region": "East", "seed": 8,  "team": "Ohio St.",         "record": "21-12"},
    {"region": "East", "seed": 9,  "team": "TCU",              "record": "22-11"},
    {"region": "East", "seed": 5,  "team": "St. John's",       "record": "28-6"},
    {"region": "East", "seed": 12, "team": "Northern Iowa",    "record": "23-12"},
    {"region": "East", "seed": 4,  "team": "Kansas",           "record": "23-10"},
    {"region": "East", "seed": 13, "team": "Cal Baptist",      "record": "25-8"},
    {"region": "East", "seed": 6,  "team": "Louisville",       "record": "23-10"},
    {"region": "East", "seed": 11, "team": "South Florida",    "record": "25-8"},
    {"region": "East", "seed": 3,  "team": "Michigan St.",     "record": "25-7"},
    {"region": "East", "seed": 14, "team": "North Dakota St.", "record": "27-7"},
    {"region": "East", "seed": 7,  "team": "UCLA",             "record": "23-11"},
    {"region": "East", "seed": 10, "team": "UCF",              "record": "21-11"},
    {"region": "East", "seed": 2,  "team": "UConn",            "record": "29-5"},
    {"region": "East", "seed": 15, "team": "Furman",           "record": "22-12"},

    # ===== WEST REGION =====
    {"region": "West", "seed": 1,  "team": "Arizona",          "record": "32-2"},
    {"region": "West", "seed": 16, "team": "Long Island",      "record": "24-10"},
    {"region": "West", "seed": 8,  "team": "Villanova",        "record": "24-8"},
    {"region": "West", "seed": 9,  "team": "Utah St.",         "record": "28-6"},
    {"region": "West", "seed": 5,  "team": "Wisconsin",        "record": "24-10"},
    {"region": "West", "seed": 12, "team": "High Point",       "record": "30-4"},
    {"region": "West", "seed": 4,  "team": "Arkansas",         "record": "26-8"},
    {"region": "West", "seed": 13, "team": "Hawaii",           "record": "24-8"},
    {"region": "West", "seed": 6,  "team": "BYU",              "record": "23-11"},
    # 11-seed play-in: NC State vs Texas — use placeholder, update after First Four
    {"region": "West", "seed": 11, "team": "Texas",             "record": "18-14"},
    {"region": "West", "seed": 3,  "team": "Gonzaga",          "record": "30-3"},
    {"region": "West", "seed": 14, "team": "Kennesaw St.",     "record": "21-13"},
    {"region": "West", "seed": 7,  "team": "Miami (FL)",       "record": "25-8"},
    {"region": "West", "seed": 10, "team": "Missouri",         "record": "20-12"},
    {"region": "West", "seed": 2,  "team": "Purdue",           "record": "27-8"},
    {"region": "West", "seed": 15, "team": "Queens (N.C.)",    "record": "21-13"},

    # ===== SOUTH REGION =====
    {"region": "South", "seed": 1,  "team": "Florida",         "record": "26-7"},
    # 16-seed play-in: Lehigh vs Prairie View A&M
    {"region": "South", "seed": 16, "team": "Prairie View A&M", "record": "18-17"},
    {"region": "South", "seed": 8,  "team": "Clemson",         "record": "24-10"},
    {"region": "South", "seed": 9,  "team": "Iowa",            "record": "21-12"},
    {"region": "South", "seed": 5,  "team": "Vanderbilt",      "record": "26-8"},
    {"region": "South", "seed": 12, "team": "McNeese",         "record": "28-5"},
    {"region": "South", "seed": 4,  "team": "Nebraska",        "record": "26-6"},
    {"region": "South", "seed": 13, "team": "Troy",            "record": "22-11"},
    {"region": "South", "seed": 6,  "team": "North Carolina",  "record": "24-8"},
    {"region": "South", "seed": 11, "team": "VCU",             "record": "27-7"},
    {"region": "South", "seed": 3,  "team": "Illinois",        "record": "24-8"},
    {"region": "South", "seed": 14, "team": "Penn",            "record": "18-11"},
    {"region": "South", "seed": 7,  "team": "Saint Mary's",    "record": "27-5"},
    {"region": "South", "seed": 10, "team": "Texas A&M",       "record": "21-11"},
    {"region": "South", "seed": 2,  "team": "Houston",         "record": "28-6"},
    {"region": "South", "seed": 15, "team": "Idaho",           "record": "21-14"},

    # ===== MIDWEST REGION =====
    {"region": "Midwest", "seed": 1,  "team": "Michigan",        "record": "31-3"},
    # 16-seed play-in: Howard vs UMBC
    {"region": "Midwest", "seed": 16, "team": "Howard",          "record": "24-10"},
    {"region": "Midwest", "seed": 8,  "team": "Georgia",         "record": "22-10"},
    {"region": "Midwest", "seed": 9,  "team": "Saint Louis",     "record": "28-5"},
    {"region": "Midwest", "seed": 5,  "team": "Texas Tech",      "record": "22-10"},
    {"region": "Midwest", "seed": 12, "team": "Akron",           "record": "29-5"},
    {"region": "Midwest", "seed": 4,  "team": "Alabama",         "record": "23-9"},
    {"region": "Midwest", "seed": 13, "team": "Hofstra",         "record": "24-10"},
    {"region": "Midwest", "seed": 6,  "team": "Tennessee",       "record": "22-11"},
    # 11-seed play-in: SMU vs Miami (Ohio)
    {"region": "Midwest", "seed": 11, "team": "Miami (OH)",      "record": "31-1"},
    {"region": "Midwest", "seed": 3,  "team": "Virginia",        "record": "29-5"},
    {"region": "Midwest", "seed": 14, "team": "Wright St.",      "record": "23-11"},
    {"region": "Midwest", "seed": 7,  "team": "Kentucky",        "record": "21-13"},
    {"region": "Midwest", "seed": 10, "team": "Santa Clara",     "record": "26-8"},
    {"region": "Midwest", "seed": 2,  "team": "Iowa St.",        "record": "27-7"},
    {"region": "Midwest", "seed": 15, "team": "Tennessee St.",   "record": "23-9"},
]

bracket_df = pd.DataFrame(bracket_2026)

# Parse W-L from record
def parse_record(rec):
    if rec == "TBD" or not isinstance(rec, str):
        return np.nan, np.nan
    parts = rec.split("-")
    if len(parts) == 2:
        return safe_float(parts[0]), safe_float(parts[1])
    return np.nan, np.nan

bracket_df[["wins", "losses"]] = bracket_df["record"].apply(
    lambda r: pd.Series(parse_record(r))
)
bracket_df["win_pct"] = bracket_df["wins"] / (bracket_df["wins"] + bracket_df["losses"])

# Save
bracket_df.to_json(DATA_DIR / "bracket_2026.json", orient="records", indent=2)
bracket_df.to_csv(DATA_DIR / "bracket_2026.csv", index=False)

log.info(f"2026 Bracket: {len(bracket_df)} teams across {bracket_df['region'].nunique()} regions")
print(f"\nPlay-in games ({bracket_df['play_in'].sum() if 'play_in' in bracket_df.columns else 0}):")
if "play_in" in bracket_df.columns:
    display(bracket_df[bracket_df["play_in"] == True][["region", "seed", "team", "play_in_teams"]])

print(f"\nFull bracket:")
display(bracket_df[["region", "seed", "team", "record", "win_pct"]].to_string(index=False))

2026-03-18 21:54:02,884 [INFO] 2026 Bracket: 64 teams across 4 regions



Play-in games (0):

Full bracket:


" region  seed             team record  win_pct\n   East     1             Duke   32-2 0.941176\n   East    16            Siena  23-11 0.676471\n   East     8         Ohio St.  21-12 0.636364\n   East     9              TCU  22-11 0.666667\n   East     5       St. John's   28-6 0.823529\n   East    12    Northern Iowa  23-12 0.657143\n   East     4           Kansas  23-10 0.696970\n   East    13      Cal Baptist   25-8 0.757576\n   East     6       Louisville  23-10 0.696970\n   East    11    South Florida   25-8 0.757576\n   East     3     Michigan St.   25-7 0.781250\n   East    14 North Dakota St.   27-7 0.794118\n   East     7             UCLA  23-11 0.676471\n   East    10              UCF  21-11 0.656250\n   East     2            UConn   29-5 0.852941\n   East    15           Furman  22-12 0.647059\n   West     1          Arizona   32-2 0.941176\n   West    16      Long Island  24-10 0.705882\n   West     8        Villanova   24-8 0.750000\n   West     9         Utah St.   28-6 0

In [10]:
# ============================================================
# 1.9 FEATURE ENGINEERING — 2026 TOURNAMENT TEAMS
# ============================================================
log.info("Building feature matrix for 2026 tournament teams...")

# Start with bracket as the base
features = bracket_df[["region", "seed", "team", "record", "wins", "losses", "win_pct"]].copy()

# --- Build SRef lookup (lowercase → original name) ---
sref_basic_lookup = {}
sref_adv_lookup = {}

if sref_basic is not None:
    sref_basic_lookup = {clean_team_name(s): s for s in sref_basic["School"].values}
if sref_adv is not None:
    sref_adv_lookup = {clean_team_name(s): s for s in sref_adv["School"].values}

# --- Match each bracket team to SRef ---
sref_basic_matches = {}
sref_adv_matches = {}
unmatched = []

for _, row in features.iterrows():
    team = row["team"]
    
    basic_match = match_team(team, sref_basic_lookup)
    if basic_match:
        sref_basic_matches[team] = basic_match
    
    adv_match = match_team(team, sref_adv_lookup)
    if adv_match:
        sref_adv_matches[team] = adv_match
    
    if not basic_match:
        unmatched.append(team)

log.info(f"  SRef basic matched: {len(sref_basic_matches)}/{len(features)}")
log.info(f"  SRef adv matched: {len(sref_adv_matches)}/{len(features)}")
if unmatched:
    log.warning(f"  Still unmatched: {unmatched}")

# --- Merge SRef basic stats ---
basic_stat_cols = ["SRS", "SOS", "FG%", "3P%", "FT%", "ORB", "TRB",
                   "AST", "STL", "BLK", "TOV", "PF", "Tm.", "Opp."]

if sref_basic is not None:
    for col in basic_stat_cols:
        if col in sref_basic.columns:
            features[col] = features["team"].map(
                lambda t: sref_basic.loc[sref_basic["School"] == sref_basic_matches.get(t, ""), col].values[0]
                if sref_basic_matches.get(t) and len(sref_basic.loc[sref_basic["School"] == sref_basic_matches[t]]) > 0
                else np.nan
            )

    matched_basic = features["SRS"].notna().sum()
    log.info(f"  Merged SRef basic: {matched_basic}/{len(features)} teams with data")

# --- Merge SRef advanced stats ---
adv_stat_cols = ["Pace", "ORtg", "eFG%", "TOV%", "ORB%", "FT/FGA",
                 "FTr", "3PAr", "TS%", "TRB%", "AST%", "STL%", "BLK%"]

if sref_adv is not None:
    for col in adv_stat_cols:
        if col in sref_adv.columns:
            features[col] = features["team"].map(
                lambda t: sref_adv.loc[sref_adv["School"] == sref_adv_matches.get(t, ""), col].values[0]
                if sref_adv_matches.get(t) and len(sref_adv.loc[sref_adv["School"] == sref_adv_matches[t]]) > 0
                else np.nan
            )

    matched_adv = features["Pace"].notna().sum()
    log.info(f"  Merged SRef advanced: {matched_adv}/{len(features)} teams with data")

# --- Merge BartTorvik (if available) ---
if bart_2026 is not None and len(bart_2026) > 50:
    bt_team_col = "Team" if "Team" in bart_2026.columns else bart_2026.columns[0]

    # Skip any embedded header rows (e.g. row where Team="team")
    bt_data = bart_2026[bart_2026[bt_team_col].astype(str).str.lower() != "team"].copy()
    bt_data["_bt_clean"] = bt_data[bt_team_col].apply(clean_team_name)
    bt_lookup = {clean_team_name(t): t for t in bt_data[bt_team_col].values}

    # BT name mapping (BT uses its own naming conventions)
    # bracket name → BT name (only where they differ from TEAM_NAME_MAP)
    BT_NAME_MAP = {
        **TEAM_NAME_MAP,
        # BT uses abbreviations similar to the bracket for most teams.
        # Overrides where BT differs from SRef:
        "uconn": "connecticut",
        "vcu": "vcu",  # BT may use VCU directly
        "smu": "smu",
        "byu": "byu",
        "ucf": "ucf",
        "umbc": "umbc",
        "tcu": "tcu",
        "penn": "penn",
        "cal baptist": "cal baptist",
    }

    def match_bt_team(bracket_name):
        """Match bracket team name to BartTorvik name."""
        bn = clean_team_name(bracket_name)
        # Direct match
        if bn in bt_lookup:
            return bt_lookup[bn]
        # Try BT name map
        mapped = BT_NAME_MAP.get(bn, bn)
        if mapped in bt_lookup:
            return bt_lookup[mapped]
        # Try TEAM_NAME_MAP (SRef mapping — some overlap)
        mapped2 = TEAM_NAME_MAP.get(bn, bn)
        if mapped2 in bt_lookup:
            return bt_lookup[mapped2]
        # Substring fallback
        words = bn.replace(".", "").split()
        if words:
            for bt_clean, bt_orig in bt_lookup.items():
                if words[0] in bt_clean and len(words[0]) >= 4:
                    return bt_orig
        return None

    # Identify available BT columns worth merging (skip ID/ranking cols)
    skip_cols = {bt_team_col, "_bt_clean", "Rank", "Seed", "Record"}
    bt_stat_cols = [c for c in bt_data.columns
                    if c not in skip_cols
                    and not c.endswith("_Rank")
                    and not c.startswith("Extra_")
                    and bt_data[c].dtype != object or c == "Conf"]

    # Convert numeric columns
    for col in bt_stat_cols:
        if col != "Conf":
            bt_data[col] = pd.to_numeric(bt_data[col], errors="coerce")

    bt_unmatched = []
    for col in bt_stat_cols:
        def _get_bt(team_name, col=col):
            bt_orig = match_bt_team(team_name)
            if bt_orig is None:
                return np.nan
            row = bt_data[bt_data[bt_team_col] == bt_orig]
            if len(row) == 0:
                return np.nan
            return row.iloc[0][col]
        features[f"bt_{col}"] = features["team"].apply(_get_bt)

    # Report match quality
    first_bt_col = f"bt_{bt_stat_cols[0]}" if bt_stat_cols else None
    if first_bt_col and first_bt_col in features.columns:
        bt_matched = features[first_bt_col].notna().sum()
    else:
        bt_matched = 0

    bt_unmatched = features[features.get(first_bt_col, pd.Series(dtype=float)).isna()]["team"].tolist() if first_bt_col else []
    log.info(f"  Merged BartTorvik: {bt_matched}/{len(features)} teams, {len(bt_stat_cols)} columns")
    if bt_unmatched:
        log.warning(f"  BT unmatched: {bt_unmatched}")

# --- Engineered features ---
if "Tm." in features.columns and "Opp." in features.columns:
    features["pts_margin"] = features["Tm."] - features["Opp."]

if "SRS" in features.columns:
    features["SRS_sq"] = features["SRS"] ** 2

if "eFG%" in features.columns:
    features["efg_above_avg"] = features["eFG%"] - features["eFG%"].mean()

# Save
features.to_csv(DATA_DIR / "features_2026.csv", index=False)

log.info(f"\n✅ Feature matrix: {features.shape[0]} teams × {features.shape[1]} features")
print(f"\nColumns: {list(features.columns)}")
print(f"\nNon-null counts:")
print(features.notna().sum().to_string())
print(f"\nSample (top seeds):")
display(features[features["seed"] <= 4].sort_values(["seed", "region"]).head(16))

2026-03-18 21:54:02,917 [INFO] Building feature matrix for 2026 tournament teams...
2026-03-18 21:54:02,920 [INFO]   SRef basic matched: 64/64
2026-03-18 21:54:02,920 [INFO]   SRef adv matched: 64/64
2026-03-18 21:54:03,149 [INFO]   Merged SRef basic: 64/64 teams with data
2026-03-18 21:54:03,319 [INFO]   Merged SRef advanced: 64/64 teams with data
2026-03-18 21:54:03,848 [INFO]   Merged BartTorvik: 64/64 teams, 42 columns
2026-03-18 21:54:03,852 [INFO] 
✅ Feature matrix: 64 teams × 79 features



Columns: ['region', 'seed', 'team', 'record', 'wins', 'losses', 'win_pct', 'SRS', 'SOS', 'FG%', '3P%', 'FT%', 'ORB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'Tm.', 'Opp.', 'Pace', 'ORtg', 'eFG%', 'TOV%', 'ORB%', 'FT/FGA', 'FTr', '3PAr', 'TS%', 'TRB%', 'AST%', 'STL%', 'BLK%', 'bt_rank', 'bt_Conf', 'bt_AdjOE', 'bt_oe Rank', 'bt_AdjDE', 'bt_de Rank', 'bt_Barthag', 'bt_rank.1', 'bt_proj. W', 'bt_Proj. L', 'bt_Pro Con W', 'bt_Pro Con L', 'bt_sos', 'bt_ncsos', 'bt_consos', 'bt_Proj. SOS', 'bt_Proj. Noncon SOS', 'bt_Proj. Con SOS', 'bt_elite SOS', 'bt_elite noncon SOS', 'bt_Opp OE', 'bt_Opp DE', 'bt_Opp Proj. OE', 'bt_Opp Proj DE', 'bt_Con Adj OE', 'bt_Con Adj DE', 'bt_Qual O', 'bt_Qual D', 'bt_Qual Barthag', 'bt_Qual Games', 'bt_FUN', 'bt_ConPF', 'bt_ConPA', 'bt_ConPoss', 'bt_ConOE', 'bt_ConDE', 'bt_ConSOSRemain', 'bt_Conf Win%', 'bt_WAB', 'bt_WAB Rk', 'bt_Fun Rk', 'bt_AdjTempo', 'pts_margin', 'SRS_sq', 'efg_above_avg']

Non-null counts:
region                 64
seed                   64


,region,seed,team,record,wins,losses,win_pct,SRS,SOS,FG%,...,bt_ConDE,bt_ConSOSRemain,bt_Conf Win%,bt_WAB,bt_WAB Rk,bt_Fun Rk,bt_AdjTempo,pts_margin,SRS_sq,efg_above_avg
0,East,1,Duke,32-2,32.0,2.0,0.941176,31.55,12.41,0.490,...,0.955775,0.0,0.944444,13.612639,2,39,65.790933,651,995.4025,0.024922
48,Midwest,1,Michigan,31-3,31.0,3.0,0.911765,32.48,14.86,0.505,...,1.012586,0.0,0.950000,13.953091,1,40,71.205027,599,1054.9504,0.037922
32,South,1,Florida,26-7,26.0,7.0,0.787879,27.90,13.11,0.478,...,1.002849,0.0,0.888889,8.480105,7,253,70.385431,488,778.4100,-0.008078
16,West,1,Arizona,32-2,32.0,2.0,0.941176,29.92,12.57,0.502,...,0.985135,0.0,0.888889,13.606138,3,12,70.013361,590,895.2064,0.007922
14,East,2,UConn,29-5,29.0,5.0,0.852941,22.92,10.53,0.482,...,1.005381,0.0,0.850000,8.795612,6,41,64.571066,421,525.3264,0.009922
62,Midwest,2,Iowa St.,27-7,27.0,7.0,0.794118,27.15,10.50,0.490,...,1.017267,0.0,0.666667,7.338254,13,210,66.940510,566,737.1225,0.021922
46,South,2,Houston,28-6,28.0,6.0,0.823529,26.23,11.93,0.450,...,1.002269,0.0,0.777778,8.931423,5,171,63.315636,486,688.0129,-0.022078
30,West,2,Purdue,27-8,27.0,8.0,0.771429,25.64,14.13,0.499,...,1.118908,0.0,0.650000,9.143874,4,164,64.424606,403,657.4096,0.032922
10,East,3,Michigan St.,25-7,25.0,7.0,0.781250,23.44,12.91,0.471,...,1.033808,0.0,0.750000,7.415050,11,52,65.966596,337,549.4336,-0.007078
58,Midwest,3,Virginia,29-5,29.0,5.0,0.852941,21.60,9.36,0.463,...,1.015955,0.0,0.833333,7.693483,8,32,65.570528,416,466.5600,0.002922


In [11]:
# ============================================================
# 1.10 HISTORICAL MATCHUP FEATURES (for model training)
# ============================================================
log.info("Building historical matchup features for model training...")

hist_df = pd.read_csv(DATA_DIR / "historical_tourney.csv")


def fuzzy_bt_match(team_name, bt_names_lower):
    """
    Match an SR tournament team name to a BartTorvik team name.
    BT and SR use slightly different conventions:
      SR: "Iowa State", "St. John's (NY)", "North Carolina State"
      BT: "Iowa St.", "St. John's", "NC State"
    """
    tn = team_name.strip().lower()
    
    # Direct match
    if tn in bt_names_lower:
        return bt_names_lower[tn]
    
    # Common SR→BT transformations
    transforms = [
        # SR uses full "State", BT uses "St."
        (lambda s: s.replace(" state", " st.")),
        # SR uses "(NY)", "(FL)" etc, BT drops them
        (lambda s: re.sub(r'\s*\([^)]*\)', '', s).strip()),
        # Combine both
        (lambda s: re.sub(r'\s*\([^)]*\)', '', s).strip().replace(" state", " st.")),
        # NC State variants: SR="NC State", BT="N.C. State"
        (lambda s: s.replace("nc state", "n.c. state")),
        (lambda s: s.replace("north carolina state", "n.c. state")),
        # UNC → North Carolina
        (lambda s: "north carolina" if s == "unc" else s),
        # Ole Miss → Mississippi
        (lambda s: "mississippi" if s == "ole miss" else s),
        # UConn → Connecticut
        (lambda s: "connecticut" if s in ("uconn", "connecticut") else s),
        # VCU, SMU, UCF, TCU, UMBC
        (lambda s: "vcu" if "virginia commonwealth" in s else s),
        (lambda s: "smu" if "southern methodist" in s else s),
        (lambda s: "ucf" if "central florida" in s else s),
        (lambda s: "tcu" if "texas christian" in s else s),
        (lambda s: "umbc" if "maryland-baltimore" in s else s),
        # LIU
        (lambda s: s.replace("long island university", "long island")),
        (lambda s: s.replace(" university", "")),
        # Saint ↔ St.
        (lambda s: s.replace("saint", "st.")),
        (lambda s: s.replace("st.", "saint")),
        # Penn ↔ Pennsylvania
        (lambda s: "penn" if s == "pennsylvania" else s),
        (lambda s: "pennsylvania" if s == "penn" else s),
        # ETSU, MTSU etc
        (lambda s: "east tennessee st." if s == "etsu" else s),
        (lambda s: s.replace("east tennessee state", "east tennessee st.")),
        (lambda s: "middle tennessee" if s in ("mtsu", "middle tennessee state") else s),
        # Arkansas-Pine Bluff, Arkansas-Little Rock etc with hyphen
        (lambda s: s.replace("-", " ")),
    ]
    
    for transform in transforms:
        try:
            transformed = transform(tn)
            if transformed in bt_names_lower:
                return bt_names_lower[transformed]
        except:
            continue
    
    # Substring match: first word with 4+ chars
    words = tn.replace(".", "").replace("'", "").split()
    if words and len(words[0]) >= 4:
        for bt_low, bt_orig in bt_names_lower.items():
            if words[0] in bt_low:
                return bt_orig
    
    # Two-word match for compound names
    if len(words) >= 2:
        two_word = f"{words[0]} {words[1]}"
        for bt_low, bt_orig in bt_names_lower.items():
            if two_word in bt_low:
                return bt_orig
    
    return None


def build_matchup_features(hist_df, sref_basic_hist, sref_adv_hist, bart_historical):
    """Build pairwise features for all historical tournament games."""
    matchups = []
    bt_match_stats = {"total": 0, "matched": 0}

    for _, game in hist_df.iterrows():
        season = int(game["season"])
        s1, s2 = game.get("seed1", np.nan), game.get("seed2", np.nan)
        if pd.isna(s1) or pd.isna(s2):
            continue
        winner_idx = game.get("winner_idx", np.nan)
        if pd.isna(winner_idx):
            continue

        target = 1 if winner_idx == 1 else 0
        seed_diff = s1 - s2

        row = {
            "season": season,
            "round": game.get("round", ""),
            "round_num": game.get("round_num", 1),
            "team1": game["team1"], "team2": game["team2"],
            "seed1": s1, "seed2": s2,
            "seed_diff": seed_diff,
            "seed_sum": s1 + s2,
            "seed_diff_sq": seed_diff ** 2,
            "abs_seed_diff": abs(seed_diff),
            "higher_seed_is_team1": int(s1 < s2),
            "round_x_seeddiff": game.get("round_num", 1) * seed_diff,
            "is_upset_seed": int(s1 > s2),
            "seed1_sq": s1 ** 2, "seed2_sq": s2 ** 2,
            "seed_product": s1 * s2,
            "log_seed_ratio": np.log(max(s2, 0.5) / max(s1, 0.5)),
            "target": target,
            "score_diff": game.get("score_diff", np.nan),
        }

        # --- SRef basic diffs ---
        if season in sref_basic_hist:
            sref = sref_basic_hist[season]
            sref_lookup_low = {clean_team_name(s): idx for idx, s in enumerate(sref["School"].values)}
            t1_match = match_team(game["team1"], {k: sref.iloc[v]["School"] for k, v in sref_lookup_low.items()})
            t2_match = match_team(game["team2"], {k: sref.iloc[v]["School"] for k, v in sref_lookup_low.items()})
            t1_idx = sref_lookup_low.get(clean_team_name(t1_match)) if t1_match else None
            t2_idx = sref_lookup_low.get(clean_team_name(t2_match)) if t2_match else None

            for col in ["SRS", "SOS"]:
                if col in sref.columns and t1_idx is not None and t2_idx is not None:
                    v1, v2 = sref.iloc[t1_idx][col], sref.iloc[t2_idx][col]
                    if pd.notna(v1) and pd.notna(v2):
                        row[f"{col}_diff"] = float(v1) - float(v2)

        # --- SRef advanced diffs ---
        if season in sref_adv_hist:
            sref_a = sref_adv_hist[season]
            sref_a_lookup_low = {clean_team_name(s): idx for idx, s in enumerate(sref_a["School"].values)}
            t1_ma = match_team(game["team1"], {k: sref_a.iloc[v]["School"] for k, v in sref_a_lookup_low.items()})
            t2_ma = match_team(game["team2"], {k: sref_a.iloc[v]["School"] for k, v in sref_a_lookup_low.items()})
            t1_idx = sref_a_lookup_low.get(clean_team_name(t1_ma)) if t1_ma else None
            t2_idx = sref_a_lookup_low.get(clean_team_name(t2_ma)) if t2_ma else None

            for col in ["eFG%", "TOV%", "ORtg", "Pace", "ORB%"]:
                if col in sref_a.columns and t1_idx is not None and t2_idx is not None:
                    v1, v2 = sref_a.iloc[t1_idx][col], sref_a.iloc[t2_idx][col]
                    if pd.notna(v1) and pd.notna(v2):
                        row[f"{col}_diff"] = float(v1) - float(v2)

        # --- BartTorvik diffs ---
        if season in bart_historical:
            bt = bart_historical[season]
            bt_team_col = "Team" if "Team" in bt.columns else bt.columns[0]
            
            # Skip player-aggregated data (no AdjOE column)
            if "AdjOE" not in bt.columns:
                pass  # 2002-2007 only have AvgORtg from player aggregation
            else:
                bt_names_lower = {str(t).strip().lower(): str(t).strip() for t in bt[bt_team_col].values}
                bt_match_stats["total"] += 2
                
                t1_bt = fuzzy_bt_match(game["team1"], bt_names_lower)
                t2_bt = fuzzy_bt_match(game["team2"], bt_names_lower)
                
                if t1_bt:
                    bt_match_stats["matched"] += 1
                if t2_bt:
                    bt_match_stats["matched"] += 1
                
                for col in ["AdjOE", "AdjDE", "Barthag", "AdjTempo", "WAB"]:
                    if col in bt.columns and t1_bt and t2_bt:
                        t1_row = bt[bt[bt_team_col] == t1_bt]
                        t2_row = bt[bt[bt_team_col] == t2_bt]
                        if len(t1_row) > 0 and len(t2_row) > 0:
                            try:
                                v1, v2 = float(t1_row.iloc[0][col]), float(t2_row.iloc[0][col])
                                if pd.notna(v1) and pd.notna(v2):
                                    row[f"{col}_diff"] = v1 - v2
                            except (ValueError, TypeError):
                                pass

        # --- LEVEL FEATURES for total prediction ---
        # These are sums/averages (not diffs) — they predict game total, not spread
        if season in sref_basic_hist:
            sref = sref_basic_hist[season]
            sref_lookup_l = {clean_team_name(s): idx for idx, s in enumerate(sref["School"].values)}
            t1_m = match_team(game["team1"], {k: sref.iloc[v]["School"] for k, v in sref_lookup_l.items()})
            t2_m = match_team(game["team2"], {k: sref.iloc[v]["School"] for k, v in sref_lookup_l.items()})
            t1_i = sref_lookup_l.get(clean_team_name(t1_m)) if t1_m else None
            t2_i = sref_lookup_l.get(clean_team_name(t2_m)) if t2_m else None

            if t1_i is not None and t2_i is not None:
                for col in ["SRS"]:
                    v1, v2 = sref.iloc[t1_i].get(col), sref.iloc[t2_i].get(col)
                    if pd.notna(v1) and pd.notna(v2):
                        row[f"{col}_sum"] = float(v1) + float(v2)

        if season in sref_adv_hist:
            sref_a = sref_adv_hist[season]
            sref_a_lk = {clean_team_name(s): idx for idx, s in enumerate(sref_a["School"].values)}
            t1_ma = match_team(game["team1"], {k: sref_a.iloc[v]["School"] for k, v in sref_a_lk.items()})
            t2_ma = match_team(game["team2"], {k: sref_a.iloc[v]["School"] for k, v in sref_a_lk.items()})
            t1_ia = sref_a_lk.get(clean_team_name(t1_ma)) if t1_ma else None
            t2_ia = sref_a_lk.get(clean_team_name(t2_ma)) if t2_ma else None

            if t1_ia is not None and t2_ia is not None:
                for col in ["Pace", "ORtg"]:
                    v1, v2 = sref_a.iloc[t1_ia].get(col), sref_a.iloc[t2_ia].get(col)
                    if pd.notna(v1) and pd.notna(v2):
                        row[f"{col}_avg"] = (float(v1) + float(v2)) / 2

        if season in bart_historical:
            bt = bart_historical[season]
            bt_tc = "Team" if "Team" in bt.columns else bt.columns[0]
            if "AdjOE" in bt.columns:
                bt_nl = {str(t).strip().lower(): str(t).strip() for t in bt[bt_tc].values}
                t1b = fuzzy_bt_match(game["team1"], bt_nl)
                t2b = fuzzy_bt_match(game["team2"], bt_nl)
                if t1b and t2b:
                    t1r = bt[bt[bt_tc] == t1b]
                    t2r = bt[bt[bt_tc] == t2b]
                    if len(t1r) > 0 and len(t2r) > 0:
                        for col in ["AdjOE", "AdjDE", "AdjTempo"]:
                            if col in bt.columns:
                                try:
                                    v1, v2 = float(t1r.iloc[0][col]), float(t2r.iloc[0][col])
                                    if pd.notna(v1) and pd.notna(v2):
                                        row[f"{col}_avg"] = (v1 + v2) / 2
                                        # Cross terms for total: team1 offense vs team2 defense
                                        if col == "AdjOE":
                                            row["OE_vs_DE_1"] = v1 - float(t2r.iloc[0].get("AdjDE", v1))
                                            row["OE_vs_DE_2"] = float(t2r.iloc[0].get("AdjOE", v2)) - float(t1r.iloc[0].get("AdjDE", v2))
                                except (ValueError, TypeError):
                                    pass

        matchups.append(row)

    df = pd.DataFrame(matchups)
    
    # Report BT matching stats
    if bt_match_stats["total"] > 0:
        rate = bt_match_stats["matched"] / bt_match_stats["total"]
        log.info(f"  BT name matching: {bt_match_stats['matched']}/{bt_match_stats['total']} ({rate:.0%})")
    
    return df


matchup_df = build_matchup_features(hist_df, sref_basic_hist, sref_adv_hist, bart_historical)

if len(matchup_df) > 0:
    matchup_df.to_csv(DATA_DIR / "historical_features.csv", index=False)
    log.info(f"\n✅ Historical matchup features: {matchup_df.shape[0]} games × {matchup_df.shape[1]} features")
    print(f"\nTarget distribution: {matchup_df['target'].value_counts().to_dict()}")
    print(f"Seasons: {matchup_df['season'].min()}-{matchup_df['season'].max()}")
    print(f"\nFeature completeness:")
    print(matchup_df.notna().mean().sort_values(ascending=False).to_string())
    print(f"\nSample:")
    display(matchup_df.head(10))
else:
    log.error("No matchup features built!")

2026-03-18 21:54:03,873 [INFO] Building historical matchup features for model training...
2026-03-18 21:55:13,730 [INFO]   BT name matching: 2137/2140 (100%)
2026-03-18 21:55:13,747 [INFO] 
✅ Historical matchup features: 1448 games × 40 features



Target distribution: {1: 922, 0: 526}
Seasons: 2002-2025

Feature completeness:
season                  1.000000
round_x_seeddiff        1.000000
round                   1.000000
score_diff              1.000000
target                  1.000000
log_seed_ratio          1.000000
seed_product            1.000000
seed2_sq                1.000000
seed1_sq                1.000000
is_upset_seed           1.000000
higher_seed_is_team1    1.000000
abs_seed_diff           1.000000
seed_diff_sq            1.000000
seed_sum                1.000000
seed_diff               1.000000
seed2                   1.000000
seed1                   1.000000
team2                   1.000000
team1                   1.000000
round_num               1.000000
TOV%_diff               0.930249
eFG%_diff               0.930249
SRS_sum                 0.929558
SRS_diff                0.929558
SOS_diff                0.929558
AdjOE_diff              0.736878
AdjDE_diff              0.736878
Barthag_diff            0.73

,season,round,round_num,team1,team2,seed1,seed2,seed_diff,seed_sum,seed_diff_sq,...,AdjOE_avg,OE_vs_DE_1,OE_vs_DE_2,AdjDE_avg,AdjTempo_avg,ORtg_diff,Pace_diff,ORB%_diff,Pace_avg,ORtg_avg
0,2002,R64,1,Maryland,Siena,1.0,16.0,-15.0,17.0,225.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2002,R64,1,Wisconsin,St. John's (NY),8.0,9.0,-1.0,17.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2002,R64,1,Marquette,Tulsa,5.0,12.0,-7.0,17.0,49.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2002,R64,1,Kentucky,Valparaiso,4.0,13.0,-9.0,17.0,81.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2002,R64,1,Texas Tech,Southern Illinois,6.0,11.0,-5.0,17.0,25.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2002,R64,1,Georgia,Murray State,3.0,14.0,-11.0,17.0,121.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2002,R64,1,NC State,Michigan State,7.0,10.0,-3.0,17.0,9.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2002,R64,1,UConn,Hampton,2.0,15.0,-13.0,17.0,169.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2002,R32,2,Maryland,Wisconsin,1.0,8.0,-7.0,9.0,49.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2002,R32,2,Tulsa,Kentucky,12.0,4.0,8.0,16.0,64.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Data Pipeline Summary

### Files in `./data/`:

| File | Description |
|------|-------------|
| `bracket_2026.json` / `.csv` | Full 68-team bracket with regions, seeds, records |
| `features_2026.csv` | Merged feature matrix for all tournament teams |
| `historical_tourney.csv` | Game-level tournament results 2002–2025 (seeds, scores, winners) |
| `historical_features.csv` | Pairwise matchup features for model training |
| `barttorvik_teams_{year}.csv` | BartTorvik team efficiency data per season |
| `sref_basic_{year}.csv` | Sports Reference basic stats per season |
| `sref_adv_{year}.csv` | Sports Reference advanced stats per season |

### Key fixes from v1:
- BartTorvik now targets **team-level** data (not player-level CSV)
- Sports Reference parser handles multi-row headers and `NCAA` suffix correctly
- Historical tournament scraper correctly parses SR bracket HTML (div.winner, span for seed, a for team+score)
- 2026 bracket is **hard-coded** from official NCAA bracket (no manual editing needed)
- Historical season data scraped for model training (not just 2026)
- Play-in games flagged with `play_in=True` for updating after First Four

### After First Four (3/17-3/18):
Update the 4 play-in entries in cell 1.8:
- `NC State/Texas` → winner
- `Lehigh/PVAMU` → winner
- `Howard/UMBC` → winner
- `SMU/Miami (OH)` → winner

### Next: Open `02_modeling.ipynb`